In [1]:
from pathlib import Path
import sys
import pandas as pd
from sklearn.model_selection import train_test_split

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT))
TARGET = "fraudulent"

df = pd.read_csv(ROOT / "data" / "raw" / "fake_job_postings.csv")
df = df.loc[df[TARGET].isin([0, 1])].copy()
df = df.drop(columns=["job_id"])
df = df.drop_duplicates().reset_index(drop=True)

X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42)

print(len(X_train), len(X_val), len(X_test))
print(y_train.sum(), y_val.sum(), y_test.sum())

12319 2640 2640
599 129 128


In [2]:
train = X_train.copy()
train[TARGET] = y_train

# 1. How much is missing in each column?
print((train.isnull().mean() * 100).round(1).sort_values(ascending=False))

# 2. Does a missing field relate to fraud?
for col in ["company_profile", "requirements", "benefits", "department"]:
    miss = train[col].isnull()
    print(f"{col}: fraud rate if missing = {train.loc[miss, TARGET].mean():.3f}, "
          f"if present = {train.loc[~miss, TARGET].mean():.3f}")

# 3. Are fake postings shorter or longer?
for col in ["title", "company_profile", "description", "requirements", "benefits"]:
    length = train[col].fillna("").str.len()
    print(f"{col}: median length real = {length[train[TARGET]==0].median():.0f}, "
          f"fraud = {length[train[TARGET]==1].median():.0f}")

# 4. Read 3 fraud descriptions
for text in train.loc[train[TARGET] == 1, "description"].head(3):
    print("\n---\n", text[:400])

salary_range           84.0
department             64.3
required_education     45.3
benefits               40.4
required_experience    39.5
function               36.2
industry               27.1
employment_type        19.4
company_profile        18.6
requirements           15.0
location                2.0
title                   0.0
description             0.0
telecommuting           0.0
has_questions           0.0
has_company_logo        0.0
fraudulent              0.0
dtype: float64
company_profile: fraud rate if missing = 0.175, if present = 0.020
requirements: fraud rate if missing = 0.059, if present = 0.047
benefits: fraud rate if missing = 0.050, if present = 0.048
department: fraud rate if missing = 0.046, if present = 0.053
title: median length real = 25, fraud = 27
company_profile: median length real = 586, fraud = 0
description: median length real = 1026, fraud = 821
requirements: median length real = 478, fraud = 249
benefits: median length real = 46, fraud = 41

---
 Cign

In [3]:
from src.preprocessing import clean_input, build_preprocessor, RAW_INPUT_COLUMNS

X_train_ready = clean_input(X_train)
X_val_ready = clean_input(X_val)
# X_test is NOT touched here. It stays unused until the final evaluation.

preprocessor = build_preprocessor()
Xt_train = preprocessor.fit_transform(X_train_ready)   # learns from train only
Xt_val = preprocessor.transform(X_val_ready)           # only applies what it learned
print(Xt_train.shape, Xt_val.shape)

(12319, 53840) (2640, 53840)


In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (precision_score, recall_score, f1_score,
                             average_precision_score, confusion_matrix)

baseline = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
baseline.fit(Xt_train, y_train)

val_pred = baseline.predict(Xt_val)
val_prob = baseline.predict_proba(Xt_val)[:, 1]

print("Precision:", round(precision_score(y_val, val_pred), 3))
print("Recall:   ", round(recall_score(y_val, val_pred), 3))
print("F1:       ", round(f1_score(y_val, val_pred), 3))
print("PR-AUC:   ", round(average_precision_score(y_val, val_prob), 3))
print(confusion_matrix(y_val, val_pred))

Precision: 0.768
Recall:    0.899
F1:        0.829
PR-AUC:    0.926
[[2476   35]
 [  13  116]]


In [5]:
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

candidates = {
    "Logistic Regression": baseline,
    "Linear SVM": LinearSVC(class_weight="balanced", dual=False, random_state=42),
    "Random Forest": RandomForestClassifier(
        n_estimators=200, class_weight="balanced_subsample",
        n_jobs=-1, random_state=42),
}

rows = []
for name, model in candidates.items():
    if name != "Logistic Regression":
        model.fit(Xt_train, y_train)
    pred = model.predict(Xt_val)
    if hasattr(model, "predict_proba"):
        score = model.predict_proba(Xt_val)[:, 1]
    else:
        score = model.decision_function(Xt_val)
    rows.append({
        "Model": name,
        "Precision": precision_score(y_val, pred),
        "Recall": recall_score(y_val, pred),
        "F1": f1_score(y_val, pred),
        "PR-AUC": average_precision_score(y_val, score),
    })

results = pd.DataFrame(rows).round(3)
print(results)

                 Model  Precision  Recall     F1  PR-AUC
0  Logistic Regression      0.768   0.899  0.829   0.926
1           Linear SVM      0.954   0.798  0.869   0.945
2        Random Forest      0.988   0.628  0.768   0.909


In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

grid = [0.01, 0.1, 1, 10, 100]
tuning_rows = []

for C in grid:
    lr = LogisticRegression(C=C, max_iter=2000, class_weight="balanced", random_state=42)
    lr.fit(Xt_train, y_train)
    pred = lr.predict(Xt_val); score = lr.predict_proba(Xt_val)[:, 1]
    tuning_rows.append({"Model": "LogReg", "C": C,
        "Precision": precision_score(y_val, pred), "Recall": recall_score(y_val, pred),
        "F1": f1_score(y_val, pred), "PR-AUC": average_precision_score(y_val, score)})

    svm = LinearSVC(C=C, class_weight="balanced", dual=False, max_iter=10000, random_state=42)
    svm.fit(Xt_train, y_train)
    pred = svm.predict(Xt_val); score = svm.decision_function(Xt_val)
    tuning_rows.append({"Model": "LinearSVM", "C": C,
        "Precision": precision_score(y_val, pred), "Recall": recall_score(y_val, pred),
        "F1": f1_score(y_val, pred), "PR-AUC": average_precision_score(y_val, score)})

tuning = pd.DataFrame(tuning_rows).round(3)
print(tuning.sort_values(["Model", "C"]).to_string(index=False))

    Model      C  Precision  Recall    F1  PR-AUC
LinearSVM   0.01      0.506   0.930 0.656   0.885
LinearSVM   0.10      0.855   0.868 0.862   0.945
LinearSVM   1.00      0.954   0.798 0.869   0.945
LinearSVM  10.00      0.953   0.791 0.864   0.941
LinearSVM 100.00      0.961   0.767 0.853   0.937
   LogReg   0.01      0.277   0.899 0.423   0.593
   LogReg   0.10      0.472   0.922 0.625   0.860
   LogReg   1.00      0.768   0.899 0.829   0.926
   LogReg  10.00      0.880   0.853 0.866   0.941
   LogReg 100.00      0.914   0.822 0.865   0.936


/home/deepak/Documents/JobShield-AI/.venv/lib/python3.13/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


In [7]:
from sklearn.metrics import precision_recall_curve
import numpy as np

best_lr = LogisticRegression(C=10, max_iter=2000, class_weight="balanced", random_state=42)
best_lr.fit(Xt_train, y_train)

best_svm = LinearSVC(C=1, class_weight="balanced", dual=False, max_iter=10000, random_state=42)
best_svm.fit(Xt_train, y_train)

scores = {
    "LogReg C=10": best_lr.predict_proba(Xt_val)[:, 1],
    "LinearSVM C=1": best_svm.decision_function(Xt_val),
}

for name, s in scores.items():
    P, R, T = precision_recall_curve(y_val, s)
    f1 = 2 * P[:-1] * R[:-1] / (P[:-1] + R[:-1] + 1e-12)
    i = f1.argmax()
    ok = T[f1 >= f1.max() - 0.01]
    print(f"{name}: best threshold={T[i]:.3f}  precision={P[i]:.3f}  "
          f"recall={R[i]:.3f}  F1={f1[i]:.3f}")
    print(f"   thresholds within 0.01 of best F1: {ok.min():.3f} to {ok.max():.3f}")

LogReg C=10: best threshold=0.569  precision=0.915  recall=0.837  F1=0.874
   thresholds within 0.01 of best F1: 0.452 to 0.588
LinearSVM C=1: best threshold=-0.228  precision=0.871  recall=0.891  F1=0.881
   thresholds within 0.01 of best F1: -0.237 to -0.086


In [8]:
s = best_lr.predict_proba(Xt_val)[:, 1]
for t in [0.4, 0.45, 0.5, 0.55, 0.6, 0.67]:
    p = (s >= t).astype(int)
    fp = int(((p == 1) & (y_val == 0)).sum())
    fn = int(((p == 0) & (y_val == 1)).sum())
    print(f"threshold={t:.2f}  precision={precision_score(y_val, p):.3f}  "
          f"recall={recall_score(y_val, p):.3f}  F1={f1_score(y_val, p):.3f}  "
          f"false alarms={fp}  missed fraud={fn}")

threshold=0.40  precision=0.826  recall=0.884  F1=0.854  false alarms=24  missed fraud=15
threshold=0.45  precision=0.851  recall=0.884  F1=0.867  false alarms=20  missed fraud=15
threshold=0.50  precision=0.880  recall=0.853  F1=0.866  false alarms=15  missed fraud=19
threshold=0.55  precision=0.908  recall=0.837  F1=0.871  false alarms=11  missed fraud=21
threshold=0.60  precision=0.912  recall=0.806  F1=0.856  false alarms=10  missed fraud=25
threshold=0.67  precision=0.936  recall=0.791  F1=0.857  false alarms=7  missed fraud=27


# Model Selection and Threshold Decision

## Metrics
Only about 4.9% of postings are fraudulent. A model that labels every posting as legitimate would reach about 95% accuracy while detecting no fraud, so accuracy is not used. Models are compared on precision, recall, F1-score and PR-AUC. PR-AUC is the main ranking metric because it does not depend on a particular decision threshold. All decisions below use the validation set only. The test set has not been used.

## 1. Candidate models (default threshold)
All models use the same preprocessing (TF-IDF on the five text fields, one-hot categorical fields, scaled salary features, binary flags), fitted on the training set only, with class weighting for the imbalance.

| Model | Precision | Recall | F1 | PR-AUC |
|---|---|---|---|---|
| Logistic Regression | 0.768 | 0.899 | 0.829 | 0.926 |
| Linear SVM | 0.954 | 0.798 | 0.869 | 0.945 |
| Random Forest | 0.988 | 0.628 | 0.768 | 0.909 |

Random Forest has very high precision but misses about 38% of fraudulent postings. It has the lowest PR-AUC and was not tuned further.

## 2. Tuning the regularization strength C

| Model | C | Precision | Recall | F1 | PR-AUC |
|---|---|---|---|---|---|
| LinearSVM | 0.01 | 0.506 | 0.930 | 0.656 | 0.885 |
| LinearSVM | 0.1 | 0.855 | 0.868 | 0.862 | 0.945 |
| LinearSVM | 1 | 0.954 | 0.798 | 0.869 | 0.945 |
| LinearSVM | 10 | 0.953 | 0.791 | 0.864 | 0.941 |
| LinearSVM | 100 | 0.961 | 0.767 | 0.853 | 0.937 |
| LogReg | 0.01 | 0.277 | 0.899 | 0.423 | 0.593 |
| LogReg | 0.1 | 0.472 | 0.922 | 0.625 | 0.860 |
| LogReg | 1 | 0.768 | 0.899 | 0.829 | 0.926 |
| LogReg | 10 | 0.880 | 0.853 | 0.866 | 0.941 |
| LogReg | 100 | 0.941 | 0.822 | 0.865 | 0.936 |

Best settings by validation PR-AUC: C = 10 for Logistic Regression (0.941) and C = 1 for Linear SVM (0.946). The validation set contains only 129 fraudulent postings, so a difference of about 0.005 PR-AUC is within noise and does not show that one model is truly better.

## 3. Final model: Logistic Regression (C = 10)
Because the tuned models performed equivalently, the choice rests on practical requirements:
- It outputs a true probability between 0 and 1, which the application can show directly as a risk score. The SVM produces an uncalibrated score.
- Its coefficients show which words and features push a prediction toward fraud, which supports explanations based on actual model evidence.
- It is simpler and faster to train and deploy.

## 4. Decision threshold

| Threshold | Precision | Recall | F1 | False alarms | Missed fraud |
|---|---|---|---|---|---|
| 0.40 | 0.826 | 0.884 | 0.854 | 24 | 15 |
| 0.45 | 0.851 | 0.884 | 0.867 | 20 | 15 |
| 0.50 | 0.880 | 0.853 | 0.866 | 15 | 19 |
| 0.55 | 0.908 | 0.837 | 0.871 | 11 | 21 |
| 0.60 | 0.912 | 0.806 | 0.856 | 10 | 25 |
| 0.67 | 0.936 | 0.791 | 0.857 | 7 | 27 |

F1 is nearly flat between 0.40 and 0.67 (0.860 to 0.866), so F1 alone cannot decide the threshold. The choice therefore rests on the cost of each error. A missed fraudulent posting can lead a job seeker to a scam, while a false alarm only leads a user to double-check a genuine posting. **Chosen threshold: 0.45** At a threshold of 0.45, the model achieves 0.851 precision, 0.884 recall and 0.867 F1-score on the validation set. It produces 20 false alarms and misses 15 fraudulent postings.

We selected this threshold because detecting fraudulent job postings is important for protecting job seekers. A lower threshold helps catch more fraudulent postings, although it can also increase false alarms.

Compared with a threshold of 0.55, the threshold of 0.45 detects more fraudulent postings (recall of 0.884 compared with 0.837), but produces more false alarms (20 compared with 11).

Therefore, 0.45 provides a reasonable balance between fraud detection and false alarms.

## Limitations
The threshold and C were chosen using the same validation set, so validation scores are slightly optimistic. Unbiased performance is reported only from the untouched test set (to be added after final evaluation).

In [9]:
CHOSEN_MODEL = "LogisticRegression"
CHOSEN_C = 10
THRESHOLD = 0.45
print(CHOSEN_MODEL, CHOSEN_C, THRESHOLD)

LogisticRegression 10 0.45


In [10]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from src.preprocessing import clean_input, build_preprocessor, RAW_INPUT_COLUMNS

final_pipeline = Pipeline([
    ("clean", FunctionTransformer(clean_input)),
    ("features", build_preprocessor()),
    ("model", LogisticRegression(C=CHOSEN_C, max_iter=2000,
                                 class_weight="balanced", random_state=42)),
])

final_pipeline.fit(X_train[RAW_INPUT_COLUMNS], y_train)

val_prob = final_pipeline.predict_proba(X_val[RAW_INPUT_COLUMNS])[:, 1]
val_pred = (val_prob >= THRESHOLD).astype(int)
print("Precision:", round(precision_score(y_val, val_pred), 3))
print("Recall:   ", round(recall_score(y_val, val_pred), 3))
print("F1:       ", round(f1_score(y_val, val_pred), 3))
print("False alarms:", int(((val_pred == 1) & (y_val == 0)).sum()))
print("Missed fraud:", int(((val_pred == 0) & (y_val == 1)).sum()))

Precision: 0.851
Recall:    0.884
F1:        0.867
False alarms: 20
Missed fraud: 15


In [11]:
import joblib
import numpy as np
from src.utils import load_model, predict_posting

MODEL_PATH = ROOT / "models" / "job_fraud_pipeline.joblib"
MODEL_PATH.parent.mkdir(exist_ok=True)
joblib.dump({"pipeline": final_pipeline, "threshold": THRESHOLD}, MODEL_PATH)

pipeline, threshold = load_model(MODEL_PATH)

# Check 1: the reloaded model gives the same probabilities
same = np.allclose(pipeline.predict_proba(X_val[RAW_INPUT_COLUMNS])[:, 1], val_prob)
print("Reloaded model matches:", same)

# Check 2: one normal-looking posting
normal = {
    "title": "Data Analyst",
    "description": "We are a team of 50 engineers building payment software. Requires SQL and Python, 2 years experience.",
    "company_profile": "Acme Payments builds secure payment tools for banks.",
    "requirements": "BSc in CS",
    "employment_type": "Full-time",
    "telecommuting": "0", "has_company_logo": "1", "has_questions": "1",
}
print("Normal:", predict_posting(normal, pipeline, threshold))

# Check 3: one scam-looking posting
scam = {
    "title": "Work from home earn $5000 weekly",
    "description": "NO EXPERIENCE NEEDED earn money fast from home",
    "salary_range": "5000-8000",
    "telecommuting": "1", "has_company_logo": "0", "has_questions": "0",
}
print("Scam:", predict_posting(scam, pipeline, threshold))

Reloaded model matches: True
Normal: {'fraud_probability': 0.0381, 'is_fraud': False, 'threshold': 0.45}
Scam: {'fraud_probability': 0.9957, 'is_fraud': True, 'threshold': 0.45}
